# 02 — Silver → Gold

Aggregates `silver.orders` (Delta) into the **`gold.vertical_performance`** business mart and writes it to the **external ADW/ALH catalog** mounted in AIDP.

**Grain**: one row per `(vertical, country_code, order_month)`.

Because the destination catalog is a federated mount to ADW/ALH, the write goes through the standard Spark catalog API — same code shape as writing to any managed catalog. The wallet and JDBC details are configured once at the workspace level; the notebook never touches them.


## Parameters


In [ ]:
# ══════════════════ Notebook config ══════════════════════════════════
# CONFIG_PATH : workspace path to assets/aidp/config.yaml. Get it from
#               the AIDP UI: right-click config.yaml → 'Copy path'.
# AIDP_ENV    : which block of config.yaml to load ('dev' or 'prod').
#               Defaults to 'dev'; Airflow can override via env var.
# ─────────────────────────────────────────────────────────────────────
import os

CONFIG_PATH = '/Workspace/sales-orders/assets/aidp/config.yaml'
AIDP_ENV    = os.environ.get('AIDP_ENV', 'dev')
print('env: ', AIDP_ENV)
# ═════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F, types as T

# === Quick-test toggle ===========================================
# Set to a small int (e.g. 5) to LIMIT the silver read for a fast
# iteration / smoke test. 0 = no limit (production behavior).
TEST_ROWS = 0
# =================================================================


In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    _cfg_all = yaml.safe_load(f)

if AIDP_ENV not in _cfg_all:
    raise KeyError(f'env "{AIDP_ENV}" not in {CONFIG_PATH}; available: {sorted(_cfg_all)}')

cfg = _cfg_all[AIDP_ENV]
print(f'env    : {AIDP_ENV}')
print(f'config : {CONFIG_PATH}')

SRC_CATALOG  = cfg['silver']['catalog']
SRC_SCHEMA   = cfg['silver']['schema']
SRC_TABLE    = cfg['silver']['table']
SRC_FQN      = f'{SRC_CATALOG}.{SRC_SCHEMA}.{SRC_TABLE}'

DEST_CATALOG = cfg['gold']['catalog']
DEST_SCHEMA  = cfg['gold']['schema']
DEST_TABLE   = cfg['gold']['table']
DEST_FQN     = f'{DEST_CATALOG}.{DEST_SCHEMA}.{DEST_TABLE}'

print(f'source : {SRC_FQN}  (delta, partitioned by country_code, order_date)')
print(f'target : {DEST_FQN}  (external catalog → ADW/ALH)')
if TEST_ROWS:
    print(f'TEST_ROWS={TEST_ROWS} — limited silver read')


## 1. Read silver — narrow projection

We only need eight columns. Delta + Catalyst already prune everything else from disk; listing them explicitly makes the intent obvious and prevents the projection from drifting when silver grows new columns.


In [ ]:
silver = (
    spark.table(SRC_FQN)
         .select(
             'vertical', 'country_code', 'order_date',
             'total_amount', 'delivery_time_minutes',
             'is_delivered', 'is_cancelled', 'is_fraudulent', 'is_prime_user',
         )
)
if TEST_ROWS:
    silver = silver.limit(TEST_ROWS)

silver.printSchema()


## 2. Aggregate — vertical performance

All metrics computed in a single `groupBy().agg()` so Spark scans the silver partitions exactly once. The boolean → int cast (`F.when(col, 1).otherwise(0)`) is Catalyst-native; no UDFs.


In [ ]:
def bool_to_int(col):
    return F.when(F.col(col), F.lit(1)).otherwise(F.lit(0))

gold = (
    silver
    .withColumn('order_month', F.trunc('order_date', 'month'))
    .groupBy('vertical', 'country_code', 'order_month')
    .agg(
        F.count('*').alias('orders'),
        F.sum(bool_to_int('is_delivered')).alias('delivered_orders'),
        F.sum(bool_to_int('is_cancelled')).alias('cancelled_orders'),
        # GMV = revenue from delivered orders only (industry convention).
        F.sum(F.when(F.col('is_delivered'), F.col('total_amount')).otherwise(F.lit(0)))
         .cast(T.DecimalType(18, 2)).alias('gmv'),
        F.avg(F.when(F.col('is_delivered'), F.col('total_amount')))
         .cast(T.DecimalType(18, 2)).alias('aov'),
        F.avg(F.when(F.col('is_delivered'), F.col('delivery_time_minutes')))
         .cast(T.DecimalType(6, 2)).alias('avg_delivery_minutes'),
        (F.sum(bool_to_int('is_cancelled')) / F.count('*'))
         .cast(T.DecimalType(5, 4)).alias('cancel_rate'),
        (F.sum(bool_to_int('is_fraudulent')) / F.count('*'))
         .cast(T.DecimalType(5, 4)).alias('fraud_rate'),
        (F.sum(bool_to_int('is_prime_user')) / F.count('*'))
         .cast(T.DecimalType(5, 4)).alias('prime_order_share'),
    )
    .withColumn('last_refreshed_at', F.current_timestamp())
)

gold.printSchema()
gold.orderBy(F.col('gmv').desc()).show(10, truncate=False)


## 3. Write to ADW/ALH via external catalog

The output mart is tiny — at most `verticals (9) × countries (9) × months (~120)` ≈ 10K rows. `coalesce(4)` keeps the JDBC fan-out into ADW under control: enough parallelism to be fast, few enough connections to not stress the DB.


In [ ]:
# `mode('overwrite').saveAsTable()` against the external ADW catalog
# creates the table on first run and replaces all rows on every
# subsequent run — no extra DDL needed from the notebook.
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {DEST_CATALOG}.{DEST_SCHEMA}')

(gold.coalesce(4)
     .write
     .mode('overwrite')
     .saveAsTable(DEST_FQN))

print(f'wrote: {DEST_FQN}')


## 4. Sanity check


In [ ]:
out = spark.table(DEST_FQN)
print(f'gold_rows={out.count()}')
out.orderBy(F.col('order_month').desc(), F.col('gmv').desc()).show(15, truncate=False)
